# Sector clustering vs statistical (correlation) clustering
**ARPM script:** `s_shrink_corr_clusters` · **Labs:** Correlation clustering, Distance matrix for correlation clustering, Clustering in the stock market

**In plain words:** do stocks that *move together* line up with their official GICS sector?

1. prices → log-returns → correlation matrix $\hat{c}^2$
2. **sector clustering:** sort $\hat{c}^2$ by sector label and look for bright diagonal blocks
3. **statistical clustering:** turn each stock into a point (simulated from $N(0,\hat{c}^2)$), run k-means, sort by the clusters found
4. **distance** $d_{ij}=\sqrt{2(1-\hat{c}^2_{ij})}$ → minimum spanning tree = the market's skeleton

Changes from ARPM's original: data path via `clustering_lab.paths`, fixed seed. Code otherwise unchanged.

In [ ]:
# --- PyCharm plumbing (not ARPM code) -------------------------------------
# Find the unit root from wherever the notebook runs, so `clustering_lab`
# imports and the data path work in PyCharm, Jupyter or a test run.
import sys
from pathlib import Path

UNIT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "clustering_lab").is_dir())
if str(UNIT_ROOT) not in sys.path:
    sys.path.insert(0, str(UNIT_ROOT))

from clustering_lab.paths import stocks_csv_path, OUTPUTS_DIR
STOCKS_CSV = stocks_csv_path()   # data/db_stocks_sp.csv (or $ARPM_DB_STOCKS_SP)
print("data:", STOCKS_CSV)

In [ ]:
from numpy import argsort, array, average, bincount, corrcoef, cumsum, \
                  diff, eye, ix_, log, max, min, r_, sqrt, zeros
from numpy.random import multivariate_normal
from networkx import draw_networkx, from_numpy_array, kamada_kawai_layout, minimum_spanning_tree
from sklearn.cluster import KMeans
from pandas import read_csv
from seaborn import heatmap, lineplot, scatterplot
from matplotlib.pyplot import show, xticks, yticks
from numpy.random import seed
seed(42)  # added: reproducible k-means / MST colouring

## Load data
Stock values with a 2-row header (sector, ticker) from `db_stocks_sp`.

In [ ]:
db_stocks_sp = read_csv(STOCKS_CSV, 
               index_col=0, header=[0, 1])
v = array(db_stocks_sp)  # stocks values
n_bar = v.shape[1]  # number of stocks
sectors = array(db_stocks_sp.columns.levels[0])  # sectors names
labels = array(db_stocks_sp.columns.codes)[0, :]  # sectors indices

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">1. Compute 
<a style="color:#3c9591" href="https://www.arpm.co/lab/redirect.php?permalink=SampCorrDeffEstim" class="@formulas #SampCorrDeffEstim">correlation</a> 
of <a style="color:#3c9591" href="https://www.arpm.co/lab/redirect.php?permalink=aosfiaod-copy-3" class="@formulas #aosfiaod-copy-3">log-returns</a> and sort it by sectors</h1>

In [ ]:
eps = diff(log(v), axis=0)  # log-returns
c2_hist = corrcoef(eps.T)  # historical correlation

i_s = argsort(labels)  # sorting index for sectors
c2_sec = c2_hist[ix_(i_s, i_s)]  # correlation matrix sorted by sectors

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">2. Compute initial clustering by computing average of each sector</h1>

In [ ]:
x = multivariate_normal(zeros(n_bar), c2_hist, 2*n_bar)  # multivariate normal scenarios
k_bar = sectors.shape[0]  # number of sectors
# compute initial clustering 
c0 = zeros((2*n_bar, k_bar))
for k in range(k_bar):
    c0[:, k] = average(x[:, labels == k], axis=1)

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">3. Determine clusters and sort correlation matrix accordingly</h1>

In [ ]:
kmeans = KMeans(n_clusters=k_bar, init=c0.T, n_init=1).fit(x.T)  # fit

i_c = argsort(kmeans.labels_)  # sorting index for clusters 
c2_clus = c2_hist[ix_(i_c, i_c)]  # correlation matrix sorted by clusters

############################################# plots #############################################
# sector clusters 
l_s = cumsum(bincount(labels[i_s]))
heatmap(c2_sec - eye(n_bar), vmin=min(c2_hist - eye(n_bar)), 
        vmax=max(c2_hist - eye(n_bar)), cmap = 'viridis', square=True)
[lineplot(x=[1, n_bar], y=[l_s[k], l_s[k]], c='r') for k in reversed(range(10))]
[lineplot(x=[l_s[k], l_s[k]], y=[1, n_bar], estimator=None, c='r') for k in reversed(range(10))]
[lineplot(x=[1, l_s[k]], y=[1, l_s[k]]) for k in reversed(range(10))]
xticks(r_[l_s[0]/2, l_s[:-1] + diff(l_s)/2], sectors, rotation=90)
yticks(r_[l_s[0]/2, l_s[:-1] + diff(l_s)/2], sectors); show();

# correlation clusters
l_c = cumsum(bincount(kmeans.labels_[i_c]))
heatmap(c2_clus - eye(n_bar), vmin=min(c2_hist - eye(n_bar)), 
        vmax=max(c2_hist - eye(n_bar)), cmap = 'viridis', square=True)
[lineplot(x=[1, n_bar], y=[l_c[k], l_c[k]], c='r') for k in reversed(range(10))]
[lineplot(x=[l_c[k], l_c[k]], y=[1, n_bar], estimator=None, c='r') for k in reversed(range(10))]
[lineplot(x=[1, l_c[k]], y=[1, l_c[k]]) for k in reversed(range(10))]
xticks([]); yticks([]); show();

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">4. Compute 
<a style="color:#3c9591" href="https://www.arpm.co/lab/redirect.php?permalink=Eucl_dist" class="@formulas #Eucl_dist">distance matrix</a> and <a style="color:#3c9591" href="https://en.wikipedia.org/wiki/Minimum_spanning_tree">minimum spanning tree</a> 
</h1>

In [ ]:
d = sqrt(2*(1 - c2_hist))  # distance matrix

mst = minimum_spanning_tree(from_numpy_array(d))  # minimum spanning tree

##################################### plots #####################################
# colors by sectors
pos = kamada_kawai_layout(mst)
draw_networkx(mst, pos=pos, node_shape='.', node_size=1, with_labels=False)
idx = [i_s[int(l_s[k - 1] if k > 0 else 0):int(l_s[k])] for k in range(k_bar)]
[scatterplot(x=[pos[id][0] for id in idx[k]] , y=[pos[id][1] for id in idx[k]])\
 for k in range(k_bar)]; show();

# colors by clusters
pos = kamada_kawai_layout(mst)
draw_networkx(mst, pos=pos, node_shape='.', node_size=1, with_labels=False)
idx = [i_c[int(l_c[k - 1] if k > 0 else 0):int(l_c[k])] for k in range(k_bar)]
[scatterplot(x=[pos[id][0] for id in idx[k]] , y=[pos[id][1] for id in idx[k]])\
 for k in range(k_bar)]; show();